# Aula de Ciência de Dados para Devs: Limpeza e Preparação de Dados

**Bem-vindo(a) à Parte 2: A Prática!**

Se na teoria tudo parece fazer sentido, é na prática que o conhecimento realmente se fixa. Nesta aula, você vai atuar como um detetive de dados. Vamos pegar um arquivo de cadastro de usuários de um e-commerce fictício (`usuarios_sujo.csv`), que está cheio de problemas comuns do mundo real, e vamos aplicar técnicas sistemáticas para limpá-lo e organizá-lo.

Ao final, teremos um dataset íntegro, confiável e pronto para a próxima fase: a análise exploratória de dados, onde poderemos extrair insights valiosos.

**Ferramentas que usaremos:**

- **Python:** Nossa linguagem de programação principal.
- **Pandas:** A biblioteca essencial para manipulação e análise de dados em Python. Pense nela como uma planilha superpoderosa que você controla com código.
- **NumPy:** Usada pelo Pandas por baixo dos panos, é fundamental para operações numéricas eficientes.
- **Faker:** Para gerar dados fictícios e criar nosso próprio dataset "sujo".


### Nosso Fluxo de Trabalho

Para nos guiar, seguiremos um fluxo de trabalho estruturado. Pense nisso como um mapa que nos levará do caos à ordem:

**1. Fonte de Dados Brutos (`usuarios_sujo.csv`)**

- Nosso ponto de partida. Um arquivo com dados realistas, porém problemáticos.

**2. Carregamento e Inspeção Inicial (Profiling)**

- Carregar os dados em um DataFrame e usar ferramentas de diagnóstico para identificar os problemas.

**3. Ciclo de Limpeza (Transformação)**

- Esta é a fase iterativa onde aplicamos as correções:
  - Tratar Valores Nulos
  - Corrigir Tipos de Dados
  - Remover Duplicatas
  - Padronizar Dados Categóricos

**4. Validação**

- Verificar se a limpeza foi bem-sucedida e se os dados agora fazem sentido.

**5. Saída de Dados Limpos (`usuarios_limpo.csv`)**

- Salvar nosso trabalho em um novo arquivo, pronto para ser usado em análises futuras.


---

## 1. Setup do Ambiente e Geração dos Dados

Primeiro, vamos importar as bibliotecas necessárias. Em seguida, vamos criar nosso próprio arquivo `usuarios_sujo.csv`. Isso garante que todos tenham exatamente o mesmo ponto de partida e que o notebook seja autocontido.

**Tipos de problemas que vamos introduzir:**

- **Valores Faltantes:** `email` e `valor_ultima_compra` terão valores nulos (`NaN`).
- **Tipos de Dados Incorretos:** `valor_ultima_compra` será uma string (object) com símbolos e vírgulas, e as colunas de data serão strings.
- **Formatos Inconsistentes:** A coluna `data_cadastro` terá múltiplos formatos de data (ex: 'YYYY-MM-DD', 'DD/MM/YYYY').
- **Dados Categóricos Não Padronizados:** A coluna `estado` terá variações como 'SP', 'São Paulo' e 'sao paulo'.
- **Linhas Duplicadas:** Inseriremos algumas linhas completamente duplicadas.


In [1]:
%pip install faker
# Importando as bibliotecas
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

# Inicializando o Faker para gerar dados em português
fake = Faker('pt_BR')

# Função para gerar os dados sujos
def gerar_dados_sujos(num_usuarios=200):
    dados = []

    # Formatos de data que vamos misturar
    formatos_data = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%d-%b-%Y']

    # Variações para os estados
    estados_sp = ['SP', 'São Paulo', 'sao paulo']
    estados_rj = ['RJ', 'Rio de Janeiro', 'rio de janeiro']
    outros_estados = ['MG', 'PR', 'BA', 'SC']

    for i in range(num_usuarios):
        # Introduzindo valores faltantes (NaN) de forma aleatória
        email = fake.email() if random.random() > 0.1 else np.nan # 10% de chance de email nulo
        valor_compra = round(random.uniform(10, 1000), 2) if random.random() > 0.15 else np.nan # 15% de chance de valor nulo

        # Formatando o valor da compra como string com inconsistências
        if pd.notna(valor_compra):
            valor_compra_str = f"R$ {valor_compra:.2f}".replace('.', ',')
        else:
            # Adicionando outras strings não numéricas para sujar mais
            valor_compra_str = np.nan if random.random() > 0.3 else 'Não informado'

        # Escolhendo um formato de data aleatório
        formato_escolhido = random.choice(formatos_data)
        data_cadastro = fake.date_between(start_date='-2y', end_date='today').strftime(formato_escolhido)

        # Escolhendo um estado com inconsistências
        if i % 4 == 0:
            estado = random.choice(estados_sp)
        elif i % 7 == 0:
            estado = random.choice(estados_rj)
        else:
            estado = random.choice(outros_estados)

        dado = {
            'user_id': fake.uuid4(),
            'nome': fake.name(),
            'email': email,
            'data_cadastro': data_cadastro,
            'cidade': fake.city(),
            'estado': estado,
            'valor_ultima_compra': valor_compra_str,
            'data_ultimo_login': fake.date_time_between(start_date='-30d', end_date='now')
        }
        dados.append(dado)

    df = pd.DataFrame(dados)

    # Introduzindo linhas duplicadas
    duplicatas = df.sample(n=15, random_state=42)
    df_final = pd.concat([df, duplicatas]).reset_index(drop=True)

    return df_final

# Gerar e salvar o arquivo CSV
df_sujo = gerar_dados_sujos()
df_sujo.to_csv('usuarios_sujo.csv', index=False)

print("Arquivo 'usuarios_sujo.csv' gerado com sucesso!")
print(f"Total de linhas: {len(df_sujo)}")

Note: you may need to restart the kernel to use updated packages.
Arquivo 'usuarios_sujo.csv' gerado com sucesso!
Total de linhas: 215


---

## 2. Passo a Passo do Exercício Prático

Agora começa a nossa missão! Vamos carregar o arquivo `usuarios_sujo.csv` que acabamos de criar e seguir nosso fluxo de trabalho para limpá-lo passo a passo.


### 2.1 Carregamento dos Dados

Usaremos a função `pd.read_csv()` do Pandas para ler nosso arquivo e carregá-lo em uma estrutura de dados chamada **DataFrame**. Pense no DataFrame como uma tabela ou planilha dentro do Python.


In [2]:
# Carregando o arquivo CSV para um DataFrame
df_usuarios = pd.read_csv('usuarios_sujo.csv')

# Exibindo as dimensões do DataFrame (linhas, colunas)
print(
    f"O dataset possui {df_usuarios.shape[0]} linhas e {df_usuarios.shape[1]} colunas.")

O dataset possui 215 linhas e 8 colunas.


### 2.2 Inspeção Inicial (Data Profiling)

Antes de sair corrigindo tudo, precisamos entender a "cena do crime". O Data Profiling é o processo de investigar o dataset para entender sua estrutura, qualidade e conteúdo. É aqui que identificamos os problemas.


#### Usando `.head()`, `.tail()` e `.sample()` para Amostragem

Essas funções nos permitem "espiar" os dados de diferentes ângulos:

- `.head(n)`: Mostra as primeiras `n` linhas (o padrão é 5).
- `.tail(n)`: Mostra as últimas `n` linhas (o padrão é 5).
- `.sample(n)`: Mostra uma amostra aleatória de `n` linhas, útil para ter uma visão imparcial do dataset.


In [3]:
print("--- Primeiras 5 linhas ---")
display(df_usuarios.head())

print("\n--- Últimas 5 linhas ---")
display(df_usuarios.tail())

print("\n--- Amostra aleatória de 5 linhas ---")
display(df_usuarios.sample(5))

--- Primeiras 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
0,c1b0385e-49d8-48a4-8ff8-f2a54c43e514,Felipe Moura,ocamara@example.net,12-Apr-2026,Rios da Serra,SP,"R$ 25,42",2026-07-26 06:23:13.785173
1,1b73e200-d37a-4843-b043-9ae729bd4d97,Srta. Lorena Dias,luiz-henrique86@example.com,18/08/2025,Rodrigues,PR,"R$ 988,42",2026-07-26 07:11:41.116929
2,78f1f0bf-b979-4cc1-96ec-c16767e339e6,Maria Isis Porto,NaN,06-25-2026,Rios,PR,"R$ 142,70",2026-08-20 01:23:54.606642
3,e2a20d0c-94c4-4a36-8f31-95903c261b46,Cecilia Pereira,pbarros@example.org,24-Oct-2024,Aparecida de Câmara,PR,NaN,2026-08-15 10:50:40.384757
4,9bf4cb24-c7e0-4efa-b37a-d5bdc6e4659e,Luiza Silveira,vitor29@example.net,20-Jan-2026,Moura de Nascimento,São Paulo,"R$ 223,49",2026-07-23 21:36:17.413636



--- Últimas 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
210,fadbfbdd-3769-42ac-89eb-4548349970d8,Maria Flor Nogueira,ecasa-grande@example.com,26/12/2025,Sá de Correia,MG,"R$ 208,39",2026-08-10 18:30:30.277600
211,858afa37-9d75-4226-91d7-aa074f35df47,Srta. Liz Dias,antoniocarvalho@example.net,02/05/2026,Cirino,rio de janeiro,"R$ 537,07",2026-08-16 14:12:52.214350
212,9cf5034a-6616-4269-bd97-ad801b806d82,Gabriela Siqueira,peixotobruno@example.org,08-30-2025,Monteiro,MG,"R$ 767,35",2026-08-03 15:27:44.728062
213,4063152c-9c10-4d57-80da-636492410593,Caroline Lima,alexandrefernandes@example.org,03-Apr-2026,Pacheco,BA,Não informado,2026-08-10 05:22:15.438518
214,358b031b-993d-4849-afa6-dfc6fd34fb58,Henrique Ramos,clararamos@example.org,10-30-2024,Ferreira do Galho,MG,"R$ 228,38",2026-08-10 12:29:17.969391



--- Amostra aleatória de 5 linhas ---


,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
71,ef6dec09-61bf-4676-87a0-09a42dd1f154,Marcelo Sá,machadoheloisa@example.com,2025-12-24,Vieira,PR,"R$ 968,29",2026-08-11 17:04:46.112831
127,090f5a6c-da32-4ece-ad6e-4d2f2abdeaae,Levi Carvalho,duartemaria-luiza@example.net,08-26-2025,da Cunha,PR,"R$ 775,34",2026-08-02 23:20:01.634053
209,d7ccd67b-d0d7-4992-909c-6b71db121954,Elisa Monteiro,vargasana-julia@example.com,21-Oct-2024,Vasconcelos,MG,"R$ 135,63",2026-08-14 06:23:20.142083
39,e8a5d73b-3f79-40b2-a700-beb5473f23b6,Sr. Leonardo Borges,barbararamos@example.net,06-12-2025,Gomes do Galho,SC,"R$ 243,34",2026-08-02 02:10:35.578359
111,0a0509da-20e3-4934-ba4c-8281cf022a58,Benício Rios,wcunha@example.com,11-02-2025,Novais de Andrade,PR,"R$ 166,36",2026-08-14 21:38:48.330318


#### Usando `.info()` para um Resumo Técnico

O método `.info()` é um dos nossos melhores amigos. Ele nos dá um resumo conciso do DataFrame, incluindo:

- O número total de entradas (linhas).
- O número de colunas.
- O nome e a contagem de valores **não nulos** para cada coluna.
- O **tipo de dado (`Dtype`)** de cada coluna.
- O uso de memória.

**O que procurar aqui?**

1.  **Contagem de Não Nulos:** Se o valor for menor que o total de entradas, significa que a coluna tem dados faltantes.
2.  **Dtype:** O tipo de dado está correto? Uma coluna de valor de compra deveria ser numérica (`float64` ou `int64`), não `object` (que geralmente significa string). Uma coluna de data deveria ser `datetime64[ns]`, não `object`.


In [4]:
# Obtendo um resumo técnico do DataFrame
df_usuarios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   user_id              215 non-null    object
 1   nome                 215 non-null    object
 2   email                186 non-null    object
 3   data_cadastro        215 non-null    object
 4   cidade               215 non-null    object
 5   estado               215 non-null    object
 6   valor_ultima_compra  193 non-null    object
 7   data_ultimo_login    215 non-null    object
dtypes: object(8)
memory usage: 13.6+ KB


#### Usando `.describe(include='all')` para Estatísticas Descritivas

Enquanto `.info()` nos dá a estrutura, `.describe()` nos dá um resumo estatístico. Usando `include='all'`, forçamos o Pandas a nos mostrar estatísticas tanto para colunas numéricas quanto para as de texto (categóricas).

- **Para colunas numéricas:** `count`, `mean` (média), `std` (desvio padrão), `min`, `max`, e os quartis (`25%`, `50%`, `75%`).
- **Para colunas de objeto/categóricas:** `count`, `unique` (número de valores únicos), `top` (valor mais frequente), e `freq` (frequência do valor mais frequente).


In [5]:
# Obtendo um resumo estatístico de todas as colunas
df_usuarios.describe(include='all')

,user_id,nome,email,data_cadastro,cidade,estado,valor_ultima_compra,data_ultimo_login
count,215,215,186,215,215,215,193,215
unique,200,200,174,191,165,10,174,200
top,d7ccd67b-d0d7-4992-909c-6b71db121954,Elisa Monteiro,limamarcelo@example.com,2024-12-18,Vasconcelos,PR,Não informado,2026-08-14 06:23:20.142083
freq,2,2,2,3,4,40,7,2


### ✏️ Exercício 1: Análise Pós-Inspeção

Com base nas saídas dos comandos `.info()` e `.describe(include='all')`, responda na célula abaixo às seguintes perguntas:

1.  Quais colunas têm valores faltantes? A contagem de não nulos em `.info()` te deu essa resposta.
2.  A coluna `valor_ultima_compra` é do tipo correto para realizarmos cálculos (como a média)?
3.  As colunas `data_cadastro` e `data_ultimo_login` estão em um formato de data que o pandas entende nativamente?
4.  Olhando para a estatística `unique` da coluna `estado` no `.describe()`, o número parece alto ou baixo demais? O que isso pode indicar?


1. As colunas que têm valores faltantes são `email` e `valor_ultima_compra`, pois a contagem de não nulos para essas colunas é menor que o total de entradas.
2. A coluna `valor_ultima_compra` não é do tipo correto para cálculos, pois está como `object` (string) devido à presença de símbolos e vírgulas.
3. As colunas `data_cadastro` e `data_ultimo_login` estão em formatos de data inconsistentes e não são reconhecidas nativamente pelo pandas como datas.
4. O número de valores únicos na coluna `estado` parece alto demais, o que pode indicar inconsistências na forma como os estados foram registrados (ex: abreviações, nomes completos, variações de maiúsculas e minúsculas).


### 2.3 Tratando Dados Faltantes (Valores Nulos)

Nossa investigação revelou que as colunas `email` e `valor_ultima_compra` têm valores faltantes. Vamos lidar com eles.


In [6]:
# Contando o número de valores nulos em cada coluna
df_usuarios.isnull().sum()

user_id                 0
nome                    0
email                  29
data_cadastro           0
cidade                  0
estado                  0
valor_ultima_compra    22
data_ultimo_login       0
dtype: int64

#### Estratégias para Lidar com Dados Faltantes

Existem duas estratégias principais para lidar com dados faltantes:

1.  **Remoção:** Excluir as linhas (ou colunas) que contêm valores faltantes. É uma abordagem rápida e simples, mas tem um custo: a perda de dados. Se uma linha tem um valor importante faltando, mas as outras informações são valiosas, removê-la pode não ser o ideal.
2.  **Imputação (Preenchimento):** Preencher os valores faltantes com um valor estimado. Pode ser um valor fixo (como 0), a média, a mediana ou a moda da coluna. Esta abordagem preserva o resto dos dados da linha, mas introduz um valor que não é original.

A escolha da estratégia depende do contexto do negócio e da natureza da coluna.


#### Estratégia 1: Remover Linhas com `dropna()`

**Cenário:** Um usuário sem e-mail é de pouca utilidade para nosso e-commerce. Não podemos contatá-lo para marketing, recuperação de senha ou confirmação de pedidos. Portanto, a decisão de negócio aqui é **remover** os cadastros que não possuem um e-mail.

Usaremos `df.dropna(subset=['nome_da_coluna'])` para remover apenas as linhas onde o valor na coluna especificada é nulo.


In [7]:
# Verificando o número de linhas antes da remoção
print(
    f"Número de linhas antes de remover nulos em 'email': {len(df_usuarios)}")

# Contando os nulos em 'email' para confirmar
print(
    f"Número de valores nulos em 'email': {df_usuarios['email'].isnull().sum()}\n")

# Removendo as linhas onde a coluna 'email' é nula
df_usuarios.dropna(subset=['email'], inplace=True)

# Verificando o número de linhas depois da remoção
print(f"Número de linhas após remover nulos em 'email': {len(df_usuarios)}")

# Confirmando que não há mais nulos em 'email'
print(
    f"Número de valores nulos em 'email' agora: {df_usuarios['email'].isnull().sum()}")

Número de linhas antes de remover nulos em 'email': 215
Número de valores nulos em 'email': 29

Número de linhas após remover nulos em 'email': 186
Número de valores nulos em 'email' agora: 0


#### Estratégia 2: Imputar Valores com `fillna()`

**Cenário:** A coluna `valor_ultima_compra` também tem valores nulos. No entanto, remover essas linhas significaria perder informações de usuários que, embora não tenham um valor de compra registrado, ainda são clientes cadastrados. Uma abordagem melhor é a **imputação**.

**Média vs. Mediana:** Qual valor usar para preencher?

- **Média (`mean`):** A soma de todos os valores dividida pelo número de valores. É muito sensível a _outliers_ (valores extremamente altos ou baixos). Uma única compra de valor muito alto poderia inflar a média e distorcer a realidade.
- **Mediana (`median`):** O valor do meio quando todos os dados são ordenados. É robusta a outliers e, por isso, é geralmente a escolha mais segura para dados financeiros ou com distribuição assimétrica, como valores de compra.

Vamos tentar calcular a mediana e preencher os valores nulos. Mas... há um problema!


In [8]:
try:
    mediana_compra = df_usuarios['valor_ultima_compra'].median()
    print(f"Mediana calculada: {mediana_compra}")
except TypeError as e:
    print(f"Ocorreu um erro: {e}")
    print("\nNão podemos calcular a mediana de uma coluna que não é numérica! Isso nos leva ao próximo passo.")

Ocorreu um erro: Cannot convert ['R$ 25,42' 'R$ 988,42' nan 'R$ 223,49' 'R$ 853,40' 'R$ 194,99'
 'R$ 432,73' 'R$ 654,11' 'R$ 384,98' 'R$ 270,07' 'R$ 190,83' 'R$ 522,92'
 'R$ 992,55' 'R$ 240,05' nan 'R$ 366,05' 'R$ 482,09' 'R$ 466,45'
 'R$ 803,70' nan 'R$ 405,62' 'R$ 521,60' 'R$ 840,05' 'R$ 180,40'
 'R$ 927,48' 'R$ 255,23' 'R$ 262,67' 'R$ 202,68' 'R$ 298,44' 'R$ 669,40'
 'R$ 47,13' 'R$ 636,06' 'R$ 688,54' 'R$ 490,18' 'R$ 243,34' 'R$ 345,52'
 'R$ 709,89' 'R$ 668,30' 'R$ 855,53' 'Não informado' 'R$ 135,63'
 'R$ 134,35' 'R$ 524,97' 'R$ 761,77' 'R$ 258,58' 'R$ 301,29' nan
 'R$ 445,70' 'R$ 285,49' 'R$ 274,95' 'R$ 365,24' nan 'R$ 469,45'
 'R$ 375,23' nan 'R$ 564,57' 'R$ 89,65' 'R$ 208,39' 'R$ 785,12'
 'R$ 805,29' 'R$ 396,24' 'R$ 968,29' 'R$ 830,54' 'R$ 840,55' 'R$ 141,22'
 'R$ 780,97' 'R$ 603,40' 'Não informado' 'R$ 729,55' 'R$ 419,68' nan
 'Não informado' 'R$ 764,30' 'R$ 435,46' 'R$ 578,19' 'R$ 661,81'
 'R$ 901,46' 'R$ 207,02' 'R$ 544,52' 'R$ 343,27' 'R$ 324,45' 'R$ 274,79'
 'R$ 135,57' 'R$ 

### 2.4 Corrigindo Tipos de Dados

O erro acima aconteceu porque, como vimos no `.info()`, a coluna `valor_ultima_compra` é do tipo `object` (string), e não um número. Precisamos convertê-la.


#### Convertendo `valor_ultima_compra` para Numérico

Para converter, precisamos primeiro limpar a string, removendo o `R$ ` e trocando a vírgula decimal por um ponto. Depois, usamos `pd.to_numeric`.

O parâmetro `errors='coerce'` é muito útil: ele transformará qualquer valor que não possa ser convertido em um número (como a string 'Não informado') em `NaN`. Isso é ótimo, pois podemos tratar todos os problemas de uma vez só.


In [9]:
# Passo 1: Limpar a string
df_usuarios['valor_ultima_compra'] = df_usuarios['valor_ultima_compra'].str.replace(
    'R$ ', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df_usuarios['valor_ultima_compra'] = df_usuarios['valor_ultima_compra'].replace(
    'Não informado', np.nan)

# Passo 2: Converter para tipo numérico, tratando erros
df_usuarios['valor_ultima_compra'] = pd.to_numeric(
    df_usuarios['valor_ultima_compra'], errors='coerce')

# Vamos verificar o tipo de dado da coluna agora
print("Tipo de dado de 'valor_ultima_compra' após conversão:")
print(df_usuarios.dtypes['valor_ultima_compra'])

# E ver como ficaram os 10 primeiros valores
print("\nValores após conversão (note os novos NaNs onde antes era 'Não informado'):")
display(df_usuarios[['nome', 'valor_ultima_compra']].head(10))

Tipo de dado de 'valor_ultima_compra' após conversão:
float64

Valores após conversão (note os novos NaNs onde antes era 'Não informado'):


,nome,valor_ultima_compra
0,Felipe Moura,25.42
1,Srta. Lorena Dias,988.42
3,Cecilia Pereira,NaN
4,Luiza Silveira,223.49
5,Ana Cavalcanti,853.40
7,João Felipe Jesus,194.99
8,Ester Ribeiro,432.73
9,Kamilly da Paz,654.11
10,Liam Nogueira,384.98
11,Amanda Vargas,270.07


#### Agora sim: Imputando a Mediana

Com a coluna `valor_ultima_compra` agora no formato `float64`, podemos finalmente calcular a mediana e usar `fillna()` para preencher todos os valores `NaN` (os que já existiam e os que foram criados pelo `errors='coerce'`).


In [10]:
# Verificando nulos ANTES da imputação
print(
    f"Nulos em 'valor_ultima_compra' ANTES da imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

# 1. Calcular a mediana (agora vai funcionar!)
mediana_compra = df_usuarios['valor_ultima_compra'].median()
print(f"Mediana calculada: {mediana_compra}")

print(f"A mediana calculada é: R$ {mediana_compra:.2f}")

# 2. Preencher os valores nulos com a mediana
df_usuarios['valor_ultima_compra'].fillna(mediana_compra, inplace=True)

# Verificando nulos depois da imputação
print(
    f"Nulos em 'valor_ultima_compra' APÓS a imputação: {df_usuarios['valor_ultima_compra'].isnull().sum()}")

Nulos em 'valor_ultima_compra' ANTES da imputação: 26
Mediana calculada: 434.095
A mediana calculada é: R$ 434.10
Nulos em 'valor_ultima_compra' APÓS a imputação: 0


/tmp/ipykernel_29662/2260482649.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_usuarios['valor_ultima_compra'].fillna(mediana_compra, inplace=True)


#### Convertendo Colunas de Data

As colunas `data_cadastro` e `data_ultimo_login` também são do tipo `object`. Precisamos convertê-las para o tipo `datetime` para que possamos realizar operações com datas, como calcular a quanto tempo um usuário se cadastrou.

A função `pd.to_datetime` é extremamente poderosa. Para a `data_cadastro`, que tem vários formatos, podemos usar o argumento `format='mixed'` para que o Pandas tente adivinhar o formato correto para cada linha.

Novamente, usaremos `errors='coerce'` para converter qualquer data que não possa ser entendida em `NaT` (Not a Time), o equivalente a `NaN` para datas.


In [11]:
# Convertendo a coluna 'data_cadastro' para datetime
# 'format="mixed"' permite que o pandas tente adivinhar múltiplos formatos
df_usuarios['data_cadastro'] = pd.to_datetime(
    df_usuarios['data_cadastro'], errors='coerce', format='mixed')

# A coluna 'data_ultimo_login' tem um formato mais consistente, mas ainda é object
df_usuarios['data_ultimo_login'] = pd.to_datetime(
    df_usuarios['data_ultimo_login'], errors='coerce')

# Vamos verificar os tipos de dados novamente com.info()
print("--- Verificação dos Dtypes após conversão de datas ---")
df_usuarios.info()

--- Verificação dos Dtypes após conversão de datas ---
<class 'pandas.core.frame.DataFrame'>
Index: 186 entries, 0 to 214
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              186 non-null    object        
 1   nome                 186 non-null    object        
 2   email                186 non-null    object        
 3   data_cadastro        186 non-null    datetime64[ns]
 4   cidade               186 non-null    object        
 5   estado               186 non-null    object        
 6   valor_ultima_compra  186 non-null    float64       
 7   data_ultimo_login    186 non-null    datetime64[ns]
dtypes: datetime64[ns](2), float64(1), object(5)
memory usage: 13.1+ KB


Sucesso! As colunas `valor_ultima_compra`, `data_cadastro` e `data_ultimo_login` agora têm os tipos de dados corretos (`float64` e `datetime64[ns]`), como esperado. Agora podemos fazer operações matemáticas e de data, como calcular o tempo desde o último login ou a média de compras.


### ✏️ Exercício 2: Cálculos Pós-Conversão

Agora que os tipos de dados estão corretos, realize as seguintes tarefas em células de código separadas:

1.  Calcule o valor **médio** da coluna `valor_ultima_compra` e imprima o resultado formatado.
2.  Encontre a data do **último login mais recente** em todo o dataset .
3.  Calcule há quantos dias foi o último login do **primeiro usuário** do DataFrame.


In [12]:
# 1. Calcule o valor médio da coluna 'valor_ultima_compra'
df_usuarios['valor_ultima_compra'].mean()

np.float64(457.9794086021506)

In [13]:
# 2. Encontre a data do último login mais recente
df_usuarios['data_ultimo_login'].max()

Timestamp('2026-08-22 14:33:40.931696')

In [14]:
# 3. Calcule há quantos dias foi o último login do primeiro usuário
dias_primeiro_usuario = (
    datetime.now() - df_usuarios['data_ultimo_login'].iloc[0]).days
print(
    f"Dias desde o último login do primeiro usuário: {dias_primeiro_usuario}")

Dias desde o último login do primeiro usuário: 27


---


### 2.5 Removendo Duplicatas

Dados duplicados podem distorcer análises, como a contagem de usuários únicos. Vamos verificar se existem e removê-los.

- `.duplicated().sum()`: Conta quantas linhas são duplicatas exatas de outras que já apareceram.
- `.drop_duplicates()`: Retorna um DataFrame com as duplicatas removidas.


In [15]:
# Verificando o número de linhas duplicadas
num_duplicatas = df_usuarios.duplicated().sum()
print(f"Número de linhas duplicadas encontradas: {num_duplicatas}")

# Removendo as duplicatas
print(f"Linhas antes de remover duplicatas: {len(df_usuarios)}")
df_usuarios.drop_duplicates(inplace=True)

print(f"Linhas após remover duplicatas: {len(df_usuarios)}")

Número de linhas duplicadas encontradas: 12
Linhas antes de remover duplicatas: 186
Linhas após remover duplicatas: 174


### 2.6 Padronizando Dados Categóricos

O último passo da nossa limpeza é garantir que dados de texto (categóricos) sejam consistentes. Na nossa inspeção, suspeitamos da coluna `estado`.

Vamos usar `.unique()` para ver todos os valores distintos que a coluna possui.


In [16]:
# Verificando os valores únicos na coluna 'estado'
print("Valores únicos em 'estado' ANTES da padronização:")

# Criando o dicionário de mapeamento para corrigir as inconsistências
mapa_estados = {
    'São Paulo': 'SP',
    'sao paulo': 'SP',
    'Rio de Janeiro': 'RJ',
    'rio de janeiro': 'RJ'
    # Não precisamos mapear 'SP' -> 'SP' ou 'RJ' -> 'RJ', o replace ignora chaves que não encontra
}

# Aplicando a substituição
df_usuarios['estado'] = df_usuarios['estado'].replace(mapa_estados)

# Verificando os valores únicos novamente para confirmar a limpeza
print("\nValores únicos em 'estado' APÓS a padronização:")
print(df_usuarios['estado'].unique())

Valores únicos em 'estado' ANTES da padronização:

Valores únicos em 'estado' APÓS a padronização:
['SP' 'PR' 'SC' 'RJ' 'MG' 'BA']


Excelente! Agora nossa coluna `estado` está limpa e padronizada, pronta para ser usada em análises de agrupamento (`groupby`).


### 2.7 Salvando o Trabalho

Missão cumprida! Passamos por todas as etapas do nosso fluxo de trabalho de limpeza. O passo final é salvar nosso DataFrame limpo em um novo arquivo CSV. Este arquivo será o ponto de partida para futuras análises.


Vamos dar uma última olhada no nosso trabalho com `.info()` para confirmar que tudo está em ordem: sem nulos e com os tipos de dados corretos.


In [17]:
# Verificação final
df_usuarios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 174 entries, 0 to 198
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   user_id              174 non-null    object        
 1   nome                 174 non-null    object        
 2   email                174 non-null    object        
 3   data_cadastro        174 non-null    datetime64[ns]
 4   cidade               174 non-null    object        
 5   estado               174 non-null    object        
 6   valor_ultima_compra  174 non-null    float64       
 7   data_ultimo_login    174 non-null    datetime64[ns]
dtypes: datetime64[ns](2), float64(1), object(5)
memory usage: 12.2+ KB


In [18]:
# Salvando o DataFrame limpo em um novo arquivo CSV
df_usuarios.to_csv('usuarios_limpo.csv', index=False)

print("Arquivo 'usuarios_limpo.csv' salvo com sucesso!")

Arquivo 'usuarios_limpo.csv' salvo com sucesso!


---

## Conclusão

Parabéns! Você completou um ciclo completo de limpeza de dados. Você pegou um dataset caótico e, aplicando um método sistemático, o transformou em uma fonte de dados organizada e confiável.

**O que nós fizemos:**

- **Inspecionamos** os dados para encontrar problemas.
- **Tratamos valores faltantes** usando duas estratégias diferentes (remoção e imputação).
- **Corrigimos tipos de dados** incorretos, permitindo cálculos e operações.
- **Removemos dados duplicados** para garantir a unicidade dos registros.
- **Padronizamos dados categóricos** para permitir agrupamentos e análises consistentes.

Agora, com o arquivo `usuarios_limpo.csv` em mãos, você está pronto para a próxima aula, onde vamos explorar e visualizar esses dados para descobrir padrões e insights de negócio.
